In [1]:
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.animation as animation
from scipy import special

import sys
sys.path.append('../')
from src.config import GROUP_PARAMS, calculate_velocity_grid, VELOCITY_SPACE, COLLISION_PARAMS

In [ ]:
Ak_data = np.load('../simulation_data/Ak_t100_20250518_232655.npy')
bk_data = np.load('../simulation_data/bk_t100_20250518_232655.npy')
wxk_data = np.load('../simulation_data/wxk_t100_20250518_232655.npy')
wyk_data = np.load('../simulation_data/wyk_t100_20250518_232655.npy')
wzk_data = np.load('../simulation_data/wzk_t100_20250518_232655.npy')
meta_data = np.load('../simulation_data/metadata_t100_20250518_232655.npy', allow_pickle=True)
print(Ak_data)

In [ ]:
cx_vec, cy_vec, cz_vec, cx, cy, cz = calculate_velocity_grid()

plt.rc('text', usetex=True)
plt.rc('font', family='serif')
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["figure.dpi"] = 150
plt.ioff()

In [ ]:
%matplotlib inline
fig = plt.figure(figsize=(6, 6))
ax1 = fig.add_subplot(111)

combined_f = np.zeros((VELOCITY_SPACE['num_cx'], VELOCITY_SPACE['num_cy'], VELOCITY_SPACE['num_cz']))
f0 = 1 / (np.pi**1.5) * np.exp(-1 * (cx**2 + cy**2 + cz**2))
# K = 1 - 0.4 * np.exp(-0/6)
# f0 = 1 / (2 * K * (np.pi * K)**1.5) * (5 * K - 3 + 2 * (1 - K) / K * (cx**2 + cy**2 + cz**2)) * np.exp(-(cx**2 + cy**2 + cz**2) / K)
f0I = np.trapz(np.trapz(f0, cz_vec, axis=2), cy_vec, axis=1)

def animate(i):
    plt.cla()
    for x in range(0, GROUP_PARAMS['num_groups_cx']):
        for j in range(0, GROUP_PARAMS['num_groups_cy']):
            for k in range(0, GROUP_PARAMS['num_groups_cz']):
                lb_cx = GROUP_PARAMS['group_bounds_cx'][x, 0]
                ub_cx = GROUP_PARAMS['group_bounds_cx'][x, 1]
                lb_cy = GROUP_PARAMS['group_bounds_cy'][j, 0]
                ub_cy = GROUP_PARAMS['group_bounds_cy'][j, 1]
                lb_cz = GROUP_PARAMS['group_bounds_cz'][k, 0]
                ub_cz = GROUP_PARAMS['group_bounds_cz'][k, 1]

                f = Ak_data[i, x, j, k] * np.exp(-bk_data[i, x, j, k] * \
                                                 ((cx[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz] - wxk_data[i, x, j, k])**2 + \
                                                  (cy[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz] - wyk_data[i, x, j, k])**2 + \
                                                    (cz[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz] - wzk_data[i, x, j, k])**2))

                combined_f[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz] = f

    combined_f_Ix = np.trapz(np.trapz(combined_f, cz_vec, axis=2), cy_vec, axis=1)
    ax1.plot(cx_vec[0:81], combined_f_Ix[0:81], color='blue')
    ax1.plot(cx_vec[80:121], combined_f_Ix[80:121], color='red')
    ax1.plot(cx_vec[120:161], combined_f_Ix[120:161], color='green')
    ax1.plot(cx_vec[160:241], combined_f_Ix[160:241], color='purple')
    ax1.set_ylim(-0.02, 0.6)
    ax1.set_xlabel(r'$C_x$', fontsize=20)
    ax1.set_ylabel(r'f', fontsize=20)
    ax1.tick_params(axis='both',labelsize=16)
    ax1.plot(cx_vec, f0I, '--', color='black')

anim = animation.FuncAnimation(fig, animate, frames=len(Ak_data), interval=100)
anim.save('3d_animation.gif')



In [ ]:
Ak = Ak_data[0, 0, 0, 0]
bk = bk_data[0, 0, 0, 0]
wxk = wxk_data[0, 0, 0, 0]
wyk = wyk_data[0, 0, 0, 0]
wzk = wzk_data[0, 0, 0, 0]

I0x = np.sqrt(np.pi / (4 * bk)) * (special.erf(np.sqrt(bk) * (GROUP_PARAMS['cf_cx'][0] - wxk)) - special.erf(np.sqrt(bk) * (GROUP_PARAMS['ci_cx'][0] - wxk)))
I0y = np.sqrt(np.pi / (4 * bk)) * (special.erf(np.sqrt(bk) * (GROUP_PARAMS['cf_cy'][0] - wyk)) - special.erf(np.sqrt(bk) * (GROUP_PARAMS['ci_cy'][0] - wyk)))
I0z = np.sqrt(np.pi / (4 * bk)) * (special.erf(np.sqrt(bk) * (GROUP_PARAMS['cf_cz'][0] - wzk)) - special.erf(np.sqrt(bk) * (GROUP_PARAMS['ci_cz'][0] - wzk)))

I1x = (np.exp(-bk * (GROUP_PARAMS['ci_cx'][0] - wxk)**2) - np.exp(-bk * (GROUP_PARAMS['cf_cx'][0] - wxk)**2)) / (2 * bk)
I1y = (np.exp(-bk * (GROUP_PARAMS['ci_cy'][0] - wyk)**2) - np.exp(-bk * (GROUP_PARAMS['cf_cy'][0] - wyk)**2)) / (2 * bk)
I1z = (np.exp(-bk * (GROUP_PARAMS['ci_cz'][0] - wzk)**2) - np.exp(-bk * (GROUP_PARAMS['cf_cz'][0] - wzk)**2)) / (2 * bk)

I2x = -np.sqrt(np.pi) / (2 * np.sqrt(bk)) * \
        ((np.exp(-bk * (GROUP_PARAMS['cf_cx'][0] - wxk)**2) * (GROUP_PARAMS['cf_cx'][0] - wxk))/np.sqrt(np.pi * bk) - (np.exp(-bk * (GROUP_PARAMS['ci_cx'][0] - wxk)**2) * (GROUP_PARAMS['ci_cx'][0] - wxk))/np.sqrt(np.pi * bk)) + \
            np.sqrt(np.pi)/(4 * np.sqrt(bk**3)) * (special.erf(np.sqrt(bk) * (GROUP_PARAMS['cf_cx'][0] - wxk)) - special.erf(np.sqrt(bk) * (GROUP_PARAMS['ci_cx'][0] - wxk)))
I2y = -np.sqrt(np.pi) / (2 * np.sqrt(bk)) * \
    ((np.exp(-bk * (GROUP_PARAMS['cf_cy'][0] - wyk)**2) * (GROUP_PARAMS['cf_cy'][0] - wyk))/np.sqrt(np.pi * bk) - (np.exp(-bk * (GROUP_PARAMS['ci_cy'][0] - wyk)**2) * (GROUP_PARAMS['ci_cy'][0] - wyk))/np.sqrt(np.pi * bk)) + \
        np.sqrt(np.pi)/(4 * np.sqrt(bk**3)) * (special.erf(np.sqrt(bk) * (GROUP_PARAMS['cf_cy'][0] - wyk)) - special.erf(np.sqrt(bk) * (GROUP_PARAMS['ci_cy'][0] - wyk)))
I2z = -np.sqrt(np.pi) / (2 * np.sqrt(bk)) * \
    ((np.exp(-bk * (GROUP_PARAMS['cf_cz'][0] - wzk)**2) * (GROUP_PARAMS['cf_cz'][0] - wzk))/np.sqrt(np.pi * bk) - (np.exp(-bk * (GROUP_PARAMS['ci_cz'][0] - wzk)**2) * (GROUP_PARAMS['ci_cz'][0] - wzk))/np.sqrt(np.pi * bk)) + \
        np.sqrt(np.pi)/(4 * np.sqrt(bk**3)) * (special.erf(np.sqrt(bk) * (GROUP_PARAMS['cf_cz'][0] - wzk)) - special.erf(np.sqrt(bk) * (GROUP_PARAMS['ci_cz'][0] - wzk)))

print('density:', Ak * I0x * I0y * I0z)
print('ux:', Ak * (I1x + wxk * I0x) * I0y * I0z)
print('uy:', Ak * I0x * (I1y + wyk * I0y) * I0z)
print('uz:', Ak * I0x * I0y * (I1z + wzk * I0z))
print('energy:', (Ak * (I2x + wxk**2 * I0x + 2 * wxk * I1x) * I0y * I0z) + (Ak * (I2y + 2 * wyk * I1y + wyk**2 * I0y) * I0x * I0z) + (Ak * (I2z + 2 * wzk * I1z + wzk**2 * I0z) * I0x * I0y))

In [ ]:
entropy_list = np.zeros(len(Ak_data))
combined_f = np.zeros((VELOCITY_SPACE['num_cx'], VELOCITY_SPACE['num_cy'], VELOCITY_SPACE['num_cz']))

for t in range(len(Ak_data)):
    for i in range(GROUP_PARAMS['num_groups_cx']):
        for j in range(GROUP_PARAMS['num_groups_cy']):
            for k in range(GROUP_PARAMS['num_groups_cz']):
                lb_cx = GROUP_PARAMS['group_bounds_cx'][i, 0]
                ub_cx = GROUP_PARAMS['group_bounds_cx'][i, 1]
                lb_cy = GROUP_PARAMS['group_bounds_cy'][j, 0]
                ub_cy = GROUP_PARAMS['group_bounds_cy'][j, 1]
                lb_cz = GROUP_PARAMS['group_bounds_cz'][k, 0]
                ub_cz = GROUP_PARAMS['group_bounds_cz'][k, 1]
                
                f = Ak_data[t, i, j, k] * np.exp(-bk_data[t, i, j, k] * \
                                                 ((cx[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz] - wxk_data[t, i, j, k])**2 + \
                                                  (cy[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz] - wyk_data[t, i, j, k])**2 + \
                                                    (cz[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz] - wzk_data[t, i, j, k])**2))

                combined_f[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz] = f

    entropy_list[t] = -np.trapz(np.trapz(np.trapz(combined_f * np.log(combined_f), cz_vec, axis=2), cy_vec, axis=1), cx_vec, axis=0)

fig = plt.figure(figsize=(6, 6))
ax1 = fig.add_subplot(111)
ax1.plot(np.linspace(0, COLLISION_PARAMS['n_t'] * COLLISION_PARAMS['dt'], len(entropy_list)), entropy_list, color='black')
ax1.set_xlabel('Time', fontsize=20)
ax1.set_ylabel('Entropy', fontsize=20)
ax1.tick_params(axis='both', labelsize=16)
plt.savefig('entropy_plot.png', bbox_inches='tight')
